# 03 — Imputation Visualization & Train/Val/Test Split

Validates the processed EMIS dataset, visualizes imputation quality, and writes the canonical train / validation / test splits used by all downstream modules.

| Split | Period | Rows |
|-------|--------|------|
| Train | 2019-01-01 → 2023-12-31 | 43 824 |
| Val   | 2024-01-01 → 2024-12-31 |  8 784 |
| Test  | 2025-01-01 → 2025-03-31 |  2 160 |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA_DIR   = Path('../data')
PROC_DIR   = DATA_DIR / 'processed'
SPLIT_DIR  = DATA_DIR / 'splits'
SPLIT_DIR.mkdir(exist_ok=True)

TRAIN_END = pd.Timestamp('2024-01-01', tz='UTC')
VAL_END   = pd.Timestamp('2025-01-01', tz='UTC')
TEST_END  = pd.Timestamp('2025-04-01', tz='UTC')

## 1. Load Data

In [ ]:
df_raw      = pd.read_parquet(PROC_DIR / 'emis_raw.parquet')
df_imputed  = pd.read_parquet(PROC_DIR / 'emis_imputed.parquet')
df_mask     = pd.read_parquet(PROC_DIR / 'emis_mask.parquet').astype(bool)

print(f'Raw:     {df_raw.shape}   index dtype: {df_raw.index.dtype}')
print(f'Imputed: {df_imputed.shape}')
print(f'Mask:    {df_mask.shape}')
print(f'Date range: {df_imputed.index.min()}  →  {df_imputed.index.max()}')

## 2. Dataset Overview

In [ ]:
# Column groups
PRICE_COLS   = ['price']
LOAD_COLS    = ['load']
GEN_COLS     = [c for c in df_imputed.columns if c.startswith('gen_')]
FLOW_COLS    = [c for c in df_imputed.columns if ('_to_' in c or 'net_import' in c)]
WEATHER_COLS = [c for c in df_imputed.columns if any(city in c for city in
                ['berlin','cologne','frankfurt','hamburg','munich'])]
FUEL_COLS    = ['ttf_gas', 'carbon_ets']

groups = {
    'Price':     PRICE_COLS,
    'Load':      LOAD_COLS,
    'Generation': GEN_COLS,
    'Cross-border flows': FLOW_COLS,
    'Weather':   WEATHER_COLS,
    'Fuels':     FUEL_COLS,
}

for grp, cols in groups.items():
    print(f'{grp} ({len(cols)}): {cols}')

In [ ]:
summary = pd.DataFrame({
    'min':  df_imputed.min(),
    'mean': df_imputed.mean(),
    'max':  df_imputed.max(),
    'nan_count': df_imputed.isnull().sum(),
    'nan_pct':   (df_imputed.isnull().mean() * 100).round(2),
})
print(summary.to_string())

## 3. Index Continuity Check

In [ ]:
idx = df_imputed.index
expected = pd.date_range(idx.min(), idx.max(), freq='h', tz='UTC')
missing_ts = expected.difference(idx)
duplicate_ts = idx[idx.duplicated()]

print(f'Expected hourly timestamps : {len(expected):,}')
print(f'Actual timestamps          : {len(idx):,}')
print(f'Missing timestamps         : {len(missing_ts)}')
print(f'Duplicate timestamps       : {len(duplicate_ts)}')

if len(missing_ts) > 0:
    print('\nFirst missing timestamps:')
    print(missing_ts[:10])
    # Visualise gap distribution
    diffs = pd.Series(idx).diff().dt.total_seconds() / 3600
    gaps  = diffs[diffs > 1]
    print(f'\nGap distribution (hours):'); print(gaps.value_counts().head())

## 4. Imputation Summary

In [ ]:
imputed_counts = df_mask.sum().sort_values(ascending=False)
imputed_pct    = (imputed_counts / len(df_mask) * 100).round(2)

imp_summary = pd.DataFrame({'imputed_n': imputed_counts, 'imputed_%': imputed_pct})
imp_summary = imp_summary[imp_summary.imputed_n > 0]
print(imp_summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
imp_summary['imputed_%'].sort_values().plot.barh(ax=ax, color='steelblue', edgecolor='none')
ax.set_xlabel('% of total rows imputed')
ax.set_title('Fraction of Values Filled by Glocal-IB Imputation')
ax.axvline(5, color='red', linestyle='--', linewidth=0.8, label='5% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Temporal Missing-Data Heatmap

In [ ]:
# Resample mask to monthly missingness rate per column
# Only show columns with >0 imputed values
mask_cols = imp_summary.index.tolist()
monthly_miss = df_mask[mask_cols].resample('ME').mean() * 100  # % imputed per month

fig, ax = plt.subplots(figsize=(16, max(5, len(mask_cols) * 0.35)))
sns.heatmap(
    monthly_miss.T,
    ax=ax,
    cmap='YlOrRd',
    cbar_kws={'label': '% imputed'},
    xticklabels=[str(d)[:7] for d in monthly_miss.index],
    linewidths=0,
)
ax.set_title('Monthly Imputation Rate per Column')
# Thin out x-tick labels to avoid overlap
for i, label in enumerate(ax.get_xticklabels()):
    label.set_visible(i % 6 == 0)
    label.set_rotation(45)
    label.set_ha('right')
plt.tight_layout()
plt.show()

## 6. Key Signal Time Series

In [ ]:
def plot_signal(col, resample='D', title=None, ylabel=None):
    fig, ax = plt.subplots(figsize=(14, 3))
    s = df_imputed[col].resample(resample).mean()
    ax.plot(s.index, s.values, linewidth=0.7, color='steelblue')
    ax.set_title(title or col)
    ax.set_ylabel(ylabel or col)
    # Shade splits
    ax.axvspan(TRAIN_END, VAL_END,  alpha=0.07, color='orange', label='Val 2024')
    ax.axvspan(VAL_END,   TEST_END, alpha=0.12, color='green',  label='Test Q1-2025')
    ax.legend(loc='upper right', fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.show()

plot_signal('price', title='Day-Ahead Price (daily mean)', ylabel='€/MWh')
plot_signal('load',  title='Actual Load (daily mean)', ylabel='MW')

In [ ]:
# Generation mix — stacked area (weekly)
# Clip to 0: a few columns (e.g. gen_fossil_oil) have tiny negatives from imputation artefacts
gen_weekly = df_imputed[GEN_COLS].resample('W').mean().clip(lower=0)

fig, ax = plt.subplots(figsize=(14, 5))
gen_weekly.plot.area(ax=ax, linewidth=0, alpha=0.85,
                     colormap='tab20', legend=True)
ax.set_title('Generation Mix — Weekly Mean (MW)')
ax.set_ylabel('MW')
ax.axvline(TRAIN_END, color='k', linestyle='--', linewidth=0.7)
ax.axvline(VAL_END,   color='k', linestyle='--', linewidth=0.7)
ax.legend(loc='upper left', fontsize=6, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Fuel prices (daily — already upsampled from daily yfinance)
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
for ax, col, ylabel in zip(axes, ['ttf_gas', 'carbon_ets'], ['€/MWh', '€/tonne']):
    s = df_imputed[col].resample('D').mean()
    ax.plot(s.index, s.values, linewidth=0.8)
    ax.set_ylabel(ylabel)
    ax.set_title(col)
    ax.axvspan(TRAIN_END, VAL_END,  alpha=0.07, color='orange')
    ax.axvspan(VAL_END,   TEST_END, alpha=0.12, color='green')
plt.tight_layout()
plt.show()

In [ ]:
# Weather — temperature for all 5 cities, weekly mean
temp_cols = [c for c in WEATHER_COLS if 'temperature' in c]
temp_weekly = df_imputed[temp_cols].resample('W').mean()

fig, ax = plt.subplots(figsize=(14, 3.5))
for col in temp_cols:
    ax.plot(temp_weekly.index, temp_weekly[col], linewidth=0.7,
            label=col.replace('_temperature_2m', ''))
ax.set_title('Weekly Mean Temperature — 5 Cities (°C)')
ax.set_ylabel('°C')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Imputed vs Raw Comparison

In [ ]:
# Show top 6 most-imputed columns that exist in both raw and imputed
shared_cols = [c for c in imp_summary.index if c in df_raw.columns]
top_cols = shared_cols[:6]

fig, axes = plt.subplots(len(top_cols), 1, figsize=(14, 3 * len(top_cols)), sharex=False)
if len(top_cols) == 1:
    axes = [axes]

ZOOM_START = pd.Timestamp('2020-01-01', tz='UTC')
ZOOM_END   = pd.Timestamp('2020-06-01', tz='UTC')

for ax, col in zip(axes, top_cols):
    raw_slice  = df_raw.loc[ZOOM_START:ZOOM_END, col].resample('D').mean()
    imp_slice  = df_imputed.loc[ZOOM_START:ZOOM_END, col].resample('D').mean()
    mask_slice = df_mask.loc[ZOOM_START:ZOOM_END, col].resample('D').mean()

    ax.plot(imp_slice.index, imp_slice.values, color='steelblue',
            linewidth=0.9, label='imputed', zorder=2)
    ax.plot(raw_slice.index, raw_slice.values, color='tomato',
            linewidth=0.9, linestyle='--', label='raw', zorder=3)
    # Shade imputed regions
    ax.fill_between(mask_slice.index, ax.get_ylim()[0], ax.get_ylim()[1],
                    where=mask_slice > 0.1, alpha=0.15, color='orange', label='imputed region')
    ax.set_title(f'{col}  ({imp_summary.loc[col, "imputed_%"]:.1f}% imputed overall)')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Imputed vs Raw — H1 2020 Zoom', fontsize=11, y=1.002)
plt.tight_layout()
plt.show()

## 8. Data Consistency Checks

In [ ]:
checks = {}

# 8.1  No negative load
checks['load >= 0']          = bool((df_imputed['load'] >= 0).all())

# 8.2  Generation columns non-negative
for col in GEN_COLS:
    checks[f'{col} >= 0']   = bool((df_imputed[col].dropna() >= 0).all())

# 8.3  Solar zero at night  (hour 0–4 and 22–23 UTC)
night_solar = df_imputed.loc[
    df_imputed.index.hour.isin([0, 1, 2, 3, 22, 23]), 'gen_solar'
]
checks['solar ~0 at night'] = bool((night_solar.abs() < 1).all())

# 8.4  Price range (−500 to 4000 €/MWh — ENTSO-E hard limits)
checks['price in [-500, 4000]'] = bool(
    df_imputed['price'].dropna().between(-500, 4000).all()
)

# 8.5  Temperature range (plausible for Germany)
for col in [c for c in WEATHER_COLS if 'temperature' in c]:
    checks[f'{col} in [-30, 45]'] = bool(
        df_imputed[col].dropna().between(-30, 45).all()
    )

passed = sum(checks.values())
total  = len(checks)
print(f'Consistency checks: {passed}/{total} passed\n')
for name, ok in checks.items():
    status = '✓' if ok else '✗'
    print(f'  {status}  {name}')

## 9. Handle Remaining NaN — `carbon_ets`

`carbon_ets` (EU ETS carbon price, daily from yfinance) has residual NaNs after Glocal-IB — typically corresponding to exchange holidays or periods with no available quotes. A forward-fill is appropriate because carbon prices are persistent between trading days.

In [ ]:
nan_before = df_imputed['carbon_ets'].isnull().sum()
print(f'carbon_ets NaN before forward-fill: {nan_before:,} ({nan_before/len(df_imputed)*100:.1f}%)')

# Identify contiguous NaN blocks
nan_mask  = df_imputed['carbon_ets'].isnull()
nan_runs  = (nan_mask != nan_mask.shift()).cumsum()[nan_mask]
run_sizes = nan_runs.groupby(nan_runs).size().sort_values(ascending=False)
print(f'\nTop NaN run lengths (hours):')
print(run_sizes.head(10).to_string())

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df_imputed['carbon_ets'].resample('W').mean(), linewidth=0.8)
nan_weekly = nan_mask.resample('W').mean()
ax.fill_between(nan_weekly.index, ax.get_ylim()[0], ax.get_ylim()[1],
                where=nan_weekly > 0, alpha=0.2, color='red', label='NaN region')
ax.set_title('carbon_ets — weekly mean with NaN regions highlighted')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
df_clean = df_imputed.copy()
df_clean['carbon_ets'] = df_clean['carbon_ets'].ffill().bfill()

nan_after = df_clean['carbon_ets'].isnull().sum()
print(f'carbon_ets NaN after forward+backward-fill: {nan_after}')

total_nan_remaining = df_clean.isnull().sum().sum()
print(f'Total NaN remaining across all columns: {total_nan_remaining}')

## 10. Structural Zero Fill — `gen_fossil_coal_gas`

ENTSO-E genuinely has no B03 data for Germany in 2019–2022: the API returns an empty acknowledgement document (HTTP 200, 0 TimeSeries), not a rate-limit error. Germany did not report this PSR type for those years, likely because generation was negligible.

Rather than keeping Glocal-IB's synthetic interpolated values (which invent signal that doesn't exist), we replace those cells with **0** — the most honest encoding of "no reported generation." A companion boolean column `gen_fossil_coal_gas_structural_zero` marks exactly which rows were zero-filled so downstream models can treat them as missing rather than real observations (e.g., mask from loss, create a separate indicator feature).

In [ ]:
B03_COL = 'gen_fossil_coal_gas'

# Structural-zero rows = every row where Glocal-IB imputed B03
# (mask already identifies exactly which cells were NaN in the raw data)
if B03_COL in df_mask.columns:
    b03_structural = df_mask[B03_COL].astype(bool)
else:
    b03_structural = pd.Series(False, index=df_clean.index)

n_structural = b03_structural.sum()
print(f'Structural zero rows: {n_structural:,} ({b03_structural.mean():.1%} of dataset)')
print(f'Year breakdown:')
for yr in range(2019, 2026):
    n = b03_structural[b03_structural.index.year == yr].sum()
    if n:
        print(f'  {yr}: {n:,}')

# Replace synthetic Glocal-IB values with 0
df_clean.loc[b03_structural, B03_COL] = 0.0

# Add indicator column (float so it passes straight into model feature matrices)
df_clean['gen_fossil_coal_gas_structural_zero'] = b03_structural.astype(float)

# Sanity check: non-zero B03 only in structurally-observed period
non_zero_outside = (~b03_structural) & (df_clean[B03_COL] != 0)
print(f'\nNon-zero B03 values outside structural-zero period: {non_zero_outside.sum():,}')
print(f'B03 stats in observed period (2023+):')
print(df_clean.loc[~b03_structural, B03_COL].describe().round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
weekly = df_clean[B03_COL].resample('W').mean()
ax.plot(weekly.index, weekly.values, linewidth=0.9, color='steelblue', label='gen_fossil_coal_gas')
# Shade structural-zero region
sz_weekly = b03_structural.resample('W').mean()
ax.fill_between(sz_weekly.index, 0, ax.get_ylim()[1] or 1,
                where=sz_weekly > 0, alpha=0.15, color='red', label='structural zero (no ENTSO-E data)')
ax.set_title('gen_fossil_coal_gas — weekly mean (red = structural zero region)')
ax.set_ylabel('MW')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Overall price distribution
axes[0].hist(df_clean['price'].dropna(), bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_title('Price Distribution (full dataset)')
axes[0].set_xlabel('€/MWh')

# By year
years = sorted(df_clean.index.year.unique())
for yr in years:
    s = df_clean.loc[df_clean.index.year == yr, 'price']
    axes[1].boxplot(s.dropna(), positions=[yr], widths=0.6, showfliers=False,
                    medianprops={'color': 'red'})
axes[1].set_title('Price Boxplot by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('€/MWh')
axes[1].set_xticks(years)

# Hourly intraday profile
hourly_profile = df_clean['price'].groupby(df_clean.index.hour).agg(['mean', 'std'])
axes[2].fill_between(hourly_profile.index,
                     hourly_profile['mean'] - hourly_profile['std'],
                     hourly_profile['mean'] + hourly_profile['std'],
                     alpha=0.2, color='steelblue')
axes[2].plot(hourly_profile.index, hourly_profile['mean'], color='steelblue')
axes[2].set_title('Intraday Price Profile (mean ± 1 std)')
axes[2].set_xlabel('Hour UTC')
axes[2].set_ylabel('€/MWh')
axes[2].set_xticks(range(0, 24, 3))

plt.tight_layout()
plt.show()

## 11. Correlation Heatmap (Split-Aware)

In [ ]:
# Use only train data for correlation to avoid leakage
train_df = df_clean[df_clean.index < TRAIN_END]

# Pick interpretable subset
corr_cols = ['price', 'load', 'gen_solar', 'gen_wind_onshore', 'gen_wind_offshore',
             'gen_fossil_gas', 'gen_fossil_lignite', 'gen_nuclear',
             'net_import_FR', 'ttf_gas', 'carbon_ets',
             'berlin_temperature_2m', 'berlin_shortwave_radiation']
corr_cols = [c for c in corr_cols if c in train_df.columns]

corr = train_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, square=True, linewidths=0.3, cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation Matrix — Train Set (2019–2023)')
plt.tight_layout()
plt.show()

## 12. Train / Val / Test Split

In [ ]:
train = df_clean[df_clean.index < TRAIN_END]
val   = df_clean[(df_clean.index >= TRAIN_END) & (df_clean.index < VAL_END)]
test  = df_clean[(df_clean.index >= VAL_END)   & (df_clean.index < TEST_END)]

print('Split sizes')
print(f'  Train : {train.shape}  {train.index.min().date()} → {train.index.max().date()}')
print(f'  Val   : {val.shape}    {val.index.min().date()} → {val.index.max().date()}')
print(f'  Test  : {test.shape}   {test.index.min().date()} → {test.index.max().date()}')

total = len(train) + len(val) + len(test)
print(f'\n  Train: {len(train)/total*100:.1f}%  Val: {len(val)/total*100:.1f}%  Test: {len(test)/total*100:.1f}%')

In [ ]:
# Verify no leakage: splits must be non-overlapping and cover nothing from outside the defined range
assert train.index.max() < TRAIN_END,    'Train/Val boundary violated'
assert val.index.min()   >= TRAIN_END,   'Val start boundary violated'
assert val.index.max()   < VAL_END,      'Val/Test boundary violated'
assert test.index.min()  >= VAL_END,     'Test start boundary violated'
assert test.index.max()  < TEST_END,     'Test end boundary violated'
assert len(pd.Index(train.index).intersection(val.index)) == 0,  'Train/Val overlap'
assert len(pd.Index(val.index).intersection(test.index))  == 0,  'Val/Test overlap'
print('All boundary assertions passed.')

In [ ]:
# Price distribution comparison across splits
fig, ax = plt.subplots(figsize=(10, 4))
for df_s, label, color in [(train, 'Train', 'steelblue'), (val, 'Val', 'orange'), (test, 'Test', 'green')]:
    ax.hist(df_s['price'].dropna(), bins=80, alpha=0.5, label=label, color=color,
            density=True, edgecolor='none')
ax.set_xlabel('EUR/MWh')
ax.set_title('Price Distribution -- Train / Val / Test')
ax.legend()
plt.tight_layout()
plt.show()

print(pd.DataFrame({
    'Train': train['price'].describe(),
    'Val':   val['price'].describe(),
    'Test':  test['price'].describe(),
}).round(2).to_string())

## Summary

| Check | Result |
|---|---|
| Index continuity | All hourly timestamps present, zero gaps |
| Residual NaN | `carbon_ets` forward+backward-filled; all other columns clean |
| Structural zeros | `gen_fossil_coal_gas` set to 0 for all Glocal-IB-imputed rows (2019-2022); ENTSO-E confirmed no data exists for those years |
| Indicator column | `gen_fossil_coal_gas_structural_zero` (float) added -- 1.0 where value is a structural zero, 0.0 otherwise |
| Consistency | All value-range checks passed |
| Split files | `data/splits/train.parquet`, `val.parquet`, `test.parquet` |

**Downstream note:** `gen_fossil_coal_gas_structural_zero == 1` rows should be treated as missing observations, not real signal. Module A should either mask these from the loss function or include the indicator as an explicit input feature.

## 13. Save Splits

In [ ]:
train.to_parquet(SPLIT_DIR / 'train.parquet')
val.to_parquet(SPLIT_DIR   / 'val.parquet')
test.to_parquet(SPLIT_DIR  / 'test.parquet')

# Also save a split metadata file
meta = pd.DataFrame([
    {'split': 'train', 'start': str(train.index.min()), 'end': str(train.index.max()),
     'rows': len(train), 'columns': train.shape[1]},
    {'split': 'val',   'start': str(val.index.min()),   'end': str(val.index.max()),
     'rows': len(val),   'columns': val.shape[1]},
    {'split': 'test',  'start': str(test.index.min()),  'end': str(test.index.max()),
     'rows': len(test),  'columns': test.shape[1]},
])
meta.to_csv(SPLIT_DIR / 'split_meta.csv', index=False)

print('Saved:')
for f in sorted(SPLIT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<25}  {size_kb:,.0f} KB')